In [9]:
import json
!pip install langchain
!pip install langchain_community
!pip install jieba
!pip install modelscope
!pip install langchain_openai

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


In [30]:
zeroshot_prompt = 'A novice programmer has written a program that contains errors and fails to pass all the test cases for the problem. Please identify the potential errors in the program and specify the line numbers where the errors occur.You need to complete two steps:Identify five lines in the program that may contain potential errors.Rank these line numbers in descending order of the likelihood of the errors occurring.You should return the result in a JSON format that can be parsed by json.load. The faultyLine queue should contain five JSON objects, each with two fields:"faultyLine" (indicating the line number of the potentially faulty code),"explanation" (explaining why this line is suspected to be erroneous).'


def build_zeroshot(zeroshot_prompt, code_location):
    with open(code_location, 'r', encoding='utf-8') as file:
        code = file.read()
    prompt = zeroshot_prompt + "\n\n"
    prompt += "This is an incorrect code to the problem:\n" + code
    return prompt

In [32]:
code_location = '/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/tcas.c_indexed.txt'
prompt = build_zeroshot(zeroshot_prompt, code_location)

In [33]:
from langchain_community.retrievers import BM25Retriever
from typing import List
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import RecursiveJsonSplitter
from langchain.vectorstores import FAISS
from langchain.document_loaders import TextLoader
from langchain.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.embeddings import ModelScopeEmbeddings

In [34]:
loader = TextLoader('/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/Tutor_min.txt')
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 0,
    length_function = len,
    separators=['\n']
)
docs = text_splitter.split_documents(documents)

In [35]:
docs[0]

Document(metadata={'source': '/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/Tutor_min.txt'}, page_content='{"id":1,"incorrectCode":"1 #include <bits/stdc++.h>\\n2 using namespace std;\\n3 const int maxn = 1005;\\n4 vector <int> G[maxn];\\n5 int fa[maxn];\\n6 int main(){\\n7     freopen(\\"p.in\\", \\"r\\", stdin);\\n8     freopen(\\"p.out\\", \\"w\\", stdout);\\n9     int t, ans = 0;\\n10     cin >> t;\\n11     while(t--){\\n12         int n;\\n13         cin >> n;\\n14         for(int i = 0; i < n - 1; i++){\\n15             int u, v;\\n16             cin >> u >> v;\\n17             G[u].push_back(v);\\n18             fa[v] = u;\\n19         }\\n20       \\n21         for(int i = 1; i <= n; i++){\\n22             bool f = true;\\n23             for(int j = 0; j < G[i].size(); j++){\\n24                 if(G[i].size() < G[G[i][j]].size()){\\n25                     f = false;\\n26                     break;\\n27                 }\\n28             }\\n29             if(f && G[i].size(

In [36]:
import jieba
def preprocessing_func(text: str) -> List[str]:
    return list(jieba.cut(text))
bm25 = BM25Retriever(docs=docs,k=10)
print(bm25.k)
retriever = bm25.from_documents(docs,preprocess_func=preprocessing_func)

10


In [37]:
retriever.invoke(prompt)

[Document(metadata={'source': '/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/Tutor_min.txt'}, page_content='\n{"id":15,"incorrectCode":"1 #include<bits/stdc++.h>\\n2 const int MAXN=2e5+5;\\n3 int n;\\n4 char s[MAXN];\\n5 bool judge(){\\n6   if(n%8!=0)return false;\\n7   for(int i=1;i<=n;++i){\\n8     if(s[i]!=\'0\'&&s[i]!=\'1\'){\\n9       return false;\\n10     }\\n11   }\\n12   for(int i=1;i<=n;i+=8){ if(strncmp(s+i,\\"100\\",3)==0||strncmp(s+i,\\"110\\",3)==0)return false;\\n13                          if(s[i]==\'0\'){\\n14 if(i+8>n||s[i+8]!=\'0\') return false;\\n15                       i+=8;\\n16                         }\\n17 if(strncmp(s+i,\\"101\\",3)==0){\\n18   int tmp=0;\\n19   for(int j=i+3;j<i+8;++j){\\n20     tmp=tmp*2+(s[j]-\'0\');\\n21   }if(tmp>=26)return false;\\n22 }\\n23 }return true;\\n24 }\\n25 int main(){\\n26   freopen(\\"information.in\\",\\"r\\",stdin);\\n27   freopen(\\"information.out\\",\\"w\\",stdout);\\n28   scanf(\\"%s\\",s+1);\\n29   n=strlen(s+1);\

In [38]:
from rank_bm25 import BM25Okapi
texts = [i.page_content for i in docs]
texts_processed = [preprocessing_func(t) for t in texts]
vectorizer = BM25Okapi(texts_processed)
vectorizer.get_top_n(preprocessing_func(zeroshot_prompt),texts, n=10)

['\n{"id":15,"incorrectCode":"1 #include<bits/stdc++.h>\\n2 const int MAXN=2e5+5;\\n3 int n;\\n4 char s[MAXN];\\n5 bool judge(){\\n6   if(n%8!=0)return false;\\n7   for(int i=1;i<=n;++i){\\n8     if(s[i]!=\'0\'&&s[i]!=\'1\'){\\n9       return false;\\n10     }\\n11   }\\n12   for(int i=1;i<=n;i+=8){ if(strncmp(s+i,\\"100\\",3)==0||strncmp(s+i,\\"110\\",3)==0)return false;\\n13                          if(s[i]==\'0\'){\\n14 if(i+8>n||s[i+8]!=\'0\') return false;\\n15                       i+=8;\\n16                         }\\n17 if(strncmp(s+i,\\"101\\",3)==0){\\n18   int tmp=0;\\n19   for(int j=i+3;j<i+8;++j){\\n20     tmp=tmp*2+(s[j]-\'0\');\\n21   }if(tmp>=26)return false;\\n22 }\\n23 }return true;\\n24 }\\n25 int main(){\\n26   freopen(\\"information.in\\",\\"r\\",stdin);\\n27   freopen(\\"information.out\\",\\"w\\",stdout);\\n28   scanf(\\"%s\\",s+1);\\n29   n=strlen(s+1);\\n30   if(!judge()){\\n31     printf(\\"Error\\\\n\\");\\n32   }else{\\n33     for(int i=1;i<=n;i+=8){\\n34  

In [39]:
# embeddings = HuggingFaceEmbeddings(model_name='/home/weijiqing/miniconda3/envs/llmfl/bge-large-en-v1.5', model_kwargs = {'device': 'cuda:1'})
from modelscope import snapshot_download
model_dir = snapshot_download("AI-ModelScope/bge-large-en-v1.5", revision='master')
model_name = model_dir
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True} # set True to compute cosine similarity
embeddings = HuggingFaceBgeEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
    query_instruction="为这个句子生成表示以用于检索相关文章："
)
print(model_dir, model_name)

2024-11-06 11:09:22,751 - modelscope - WARNING - Using branch: master as version is unstable, use with caution


/home/weijiqing/.cache/modelscope/hub/AI-ModelScope/bge-large-en-v1___5 /home/weijiqing/.cache/modelscope/hub/AI-ModelScope/bge-large-en-v1___5


In [40]:
db = FAISS.from_documents(docs, embeddings)

In [41]:
db.save_local('/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data')

In [42]:
bm25_res = vectorizer.get_top_n(preprocessing_func(prompt),texts, n=10)
bm25_res

['\n{"id":15,"incorrectCode":"1 #include<bits/stdc++.h>\\n2 const int MAXN=2e5+5;\\n3 int n;\\n4 char s[MAXN];\\n5 bool judge(){\\n6   if(n%8!=0)return false;\\n7   for(int i=1;i<=n;++i){\\n8     if(s[i]!=\'0\'&&s[i]!=\'1\'){\\n9       return false;\\n10     }\\n11   }\\n12   for(int i=1;i<=n;i+=8){ if(strncmp(s+i,\\"100\\",3)==0||strncmp(s+i,\\"110\\",3)==0)return false;\\n13                          if(s[i]==\'0\'){\\n14 if(i+8>n||s[i+8]!=\'0\') return false;\\n15                       i+=8;\\n16                         }\\n17 if(strncmp(s+i,\\"101\\",3)==0){\\n18   int tmp=0;\\n19   for(int j=i+3;j<i+8;++j){\\n20     tmp=tmp*2+(s[j]-\'0\');\\n21   }if(tmp>=26)return false;\\n22 }\\n23 }return true;\\n24 }\\n25 int main(){\\n26   freopen(\\"information.in\\",\\"r\\",stdin);\\n27   freopen(\\"information.out\\",\\"w\\",stdout);\\n28   scanf(\\"%s\\",s+1);\\n29   n=strlen(s+1);\\n30   if(!judge()){\\n31     printf(\\"Error\\\\n\\");\\n32   }else{\\n33     for(int i=1;i<=n;i+=8){\\n34  

In [22]:
vector_res = db.similarity_search(prompt, k=10)
vector_res

[Document(metadata={'source': '/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/Tutor_min.txt'}, page_content='\n{"id":15,"incorrectCode":"1 #include<bits/stdc++.h>\\n2 const int MAXN=2e5+5;\\n3 int n;\\n4 char s[MAXN];\\n5 bool judge(){\\n6   if(n%8!=0)return false;\\n7   for(int i=1;i<=n;++i){\\n8     if(s[i]!=\'0\'&&s[i]!=\'1\'){\\n9       return false;\\n10     }\\n11   }\\n12   for(int i=1;i<=n;i+=8){ if(strncmp(s+i,\\"100\\",3)==0||strncmp(s+i,\\"110\\",3)==0)return false;\\n13                          if(s[i]==\'0\'){\\n14 if(i+8>n||s[i+8]!=\'0\') return false;\\n15                       i+=8;\\n16                         }\\n17 if(strncmp(s+i,\\"101\\",3)==0){\\n18   int tmp=0;\\n19   for(int j=i+3;j<i+8;++j){\\n20     tmp=tmp*2+(s[j]-\'0\');\\n21   }if(tmp>=26)return false;\\n22 }\\n23 }return true;\\n24 }\\n25 int main(){\\n26   freopen(\\"information.in\\",\\"r\\",stdin);\\n27   freopen(\\"information.out\\",\\"w\\",stdout);\\n28   scanf(\\"%s\\",s+1);\\n29   n=strlen(s+1);\

In [43]:
def rrf(vector_results: List[str], text_results: List[str], k: int=10, m: int=60):
        """
        使用RRF算法对两组检索结果进行重排序

        params:
        vector_results (list): 向量召回的结果列表,每个元素是专利ID
        text_results (list): 文本召回的结果列表,每个元素是专利ID
        k(int): 排序后返回前k个
        m (int): 超参数

        return:
        重排序后的结果列表,每个元素是(文档ID, 融合分数)
        """

        doc_scores = {}

        # 遍历两组结果,计算每个文档的融合分数
        for rank, doc_id in enumerate(vector_results):
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1 / (rank+m)
        for rank, doc_id in enumerate(text_results):
            doc_scores[doc_id] = doc_scores.get(doc_id, 0) + 1 / (rank+m)

        # 将结果按融合分数排序
        sorted_results = [d for d, _ in sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)[:k]]

        return sorted_results

In [44]:
vector_results = [i.page_content for i in vector_res]
text_results = [i for i in bm25_res]
rrf_res = rrf(vector_results, text_results)
rrf_res

['\n{"id":15,"incorrectCode":"1 #include<bits/stdc++.h>\\n2 const int MAXN=2e5+5;\\n3 int n;\\n4 char s[MAXN];\\n5 bool judge(){\\n6   if(n%8!=0)return false;\\n7   for(int i=1;i<=n;++i){\\n8     if(s[i]!=\'0\'&&s[i]!=\'1\'){\\n9       return false;\\n10     }\\n11   }\\n12   for(int i=1;i<=n;i+=8){ if(strncmp(s+i,\\"100\\",3)==0||strncmp(s+i,\\"110\\",3)==0)return false;\\n13                          if(s[i]==\'0\'){\\n14 if(i+8>n||s[i+8]!=\'0\') return false;\\n15                       i+=8;\\n16                         }\\n17 if(strncmp(s+i,\\"101\\",3)==0){\\n18   int tmp=0;\\n19   for(int j=i+3;j<i+8;++j){\\n20     tmp=tmp*2+(s[j]-\'0\');\\n21   }if(tmp>=26)return false;\\n22 }\\n23 }return true;\\n24 }\\n25 int main(){\\n26   freopen(\\"information.in\\",\\"r\\",stdin);\\n27   freopen(\\"information.out\\",\\"w\\",stdout);\\n28   scanf(\\"%s\\",s+1);\\n29   n=strlen(s+1);\\n30   if(!judge()){\\n31     printf(\\"Error\\\\n\\");\\n32   }else{\\n33     for(int i=1;i<=n;i+=8){\\n34  

In [25]:
# prompt = '''
# 任务目标：根据检索出的文档回答用户问题
# 任务要求：
#     1、不得脱离检索出的文档回答问题
#     2、若检索出的文档不包含用户问题的答案，请回答我不知道
#
# 用户问题：
# {}
#
# 检索出的文档：
# {}
# '''

In [45]:
# from langchain_openai import ChatOpenAI
from openai import OpenAI
# model = ChatOpenAI(model='chatGlm3', base_url='http://222.199.230.149:8611/v1', api_key='666')
# code_location = '/home/weijiqing/miniconda3/envs/llmfl/LLMFL/Data/tcas.c_indexed.txt'
# prompt = build_zeroshot(zeroshot_prompt, code_location)
client = OpenAI(
    base_url="http://127.0.0.1:9997/v1",
    api_key="666"
)
res = client.chat.completions.create(
    model="llama-3-instruct",
    messages=[
        {"role": "system", "content": "你是一名技术精湛的人工智能工程师和算法教师，擅长准确定位新手程序的错误。"},
        {"role": "user", "content": prompt.join(rrf_res)}
    ]
)
# res = completion.choices[0].message.content
# model = OpenAI(model='llama-3', base_url="http://127.0.0.1:9997/v1", api_key='666')
# res = model.invoke(prompt.format('从小到大', ''.join(rrf_res)))
response_txt = res.choices[0].message.content
print(response_txt)

 line numbers where the errors occur.You need to complete two steps:Identify five lines in the program that may contain potential errors.Rank these line numbers in descending order of the likelihood of the errors occurring.You should return the result in a JSON format that can be parsed by json.load. The faultyLine queue should contain five JSON objects, each with two fields:"faultyLine" (indicating the line number of the potentially faulty code),"explanation" (explaining why this line is suspected to be erroneous).

This is an incorrect code to the problem:
1 #include <stdio.h>
2 #include <stdlib.h>
3 #include<string.h>>
4 int main(int argc, char *argv[])
5 {
6     char a[1005],b[1005];
7     int ans=0,s=0,j,l,i,k=0,z;
8     gets (a);
9     l=strlen(a);
10     if (a[1]=='}')
11     {
12         printf("0");
13         return 0;
14         ans=1;
15     }
16     if (ans==0)
17     {
18         ans=1;
19         b[0]=a[1];
20         s=1;
21     for (i=4;i<l;i=i+3)
22     {
23         b

In [27]:
# res = model.invoke('从小到大')
res = client.chat.completions.create(
    model="llama-3-instruct",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": '以下这段代码的错误位置在哪里？ "#include<stdio.h>\r\nint main()\r\n{\r\n    int i,j,k,n1,n2,n3;\r\n    scanf(\"%d%d%d\",&i,&j,&k);\r\n    if(i>j)\r\n    {\r\n\r\n        if(i>k)\r\n           {\r\n            n1=j;\r\n            n2=k;\r\n            n3=i;\r\n           }\r\n        else\r\n        {\r\n             if(j>k)\r\n             {\r\n                n1=k;\r\n                n2=j;\r\n                n3=i;\r\n             }\r\n             else\r\n             {\r\n                 n1=j;\r\n                 n2=k;\r\n                 n3=i;\r\n             }\r\n        }\r\n    }\r\n   else if(i<j)\r\n    {\r\n\r\n        if(i>k)\r\n            {\r\n                n1=k;\r\n                n2=i;\r\n                n3=j;\r\n            }\r\n        else\r\n        {\r\n          if(i>k)\r\n          {\r\n               n1=i;\r\n               n2=k;\r\n               n3=j;\r\n          }\r\n          else\r\n          {\r\n                n1=i;\r\n                n2=k;\r\n                n3=j;\r\n          }\r\n        }\r\n    }\r\n    printf(\"%d %d %d\",n1,n2,n3);\r\n}\r\n"'}
    ]
)
response_txt = res.choices[0].message.content
print(response_txt)

KeyboardInterrupt: 